# <font color='blue'> Stacking (Stacked Generalization) </font>

Stacking, or **Stacked Generalization**, is an ensemble learning technique that combines the predictions of multiple different machine learning models using a second-level model known as a **meta-learner**.

Unlike Bagging and Random Forests, which combine many versions of the same learning algorithm, Stacking allows completely different models to work together. For example, a Stacking ensemble may simultaneously use

* Logistic Regression,
* Support Vector Machines,
* Random Forests,
* Gradient Boosting,
* K-Nearest Neighbours,

and then train another model to determine how these predictions should be combined.

The central philosophy is

> **Rather than deciding manually which model is best, allow another machine learning algorithm to learn the optimal combination automatically.**

Stacking is widely used in machine learning competitions because it often produces predictive performance superior to any individual model.

---



## <font color='orange'> 1. Motivation </font>

Suppose we wish to classify fraudulent credit card transactions.

Different algorithms make different kinds of mistakes.

```
Logistic Regression

↓

Simple linear decision boundary
```

```
Random Forest

↓

Excellent nonlinear relationships
```

```
Support Vector Machine

↓

Good margin separation
```

```
Gradient Boosting

↓

Captures complex interactions
```

Each model performs well,

but none is perfect.

Instead of selecting only one,

Stacking combines them all.

---



## <font color='orange'> 2. The Central Idea </font>

Stacking consists of two levels.

### Level-0 Models

These are the ordinary machine learning algorithms.

For example,

```
Logistic Regression

Support Vector Machine

Random Forest

Gradient Boosting

KNN
```

Each model produces predictions.

---

### Level-1 Model

The predictions of the Level-0 models become

the input variables

for another learning algorithm.

```
Predictions

↓

Meta-Learner

↓

Final Prediction
```

The Level-1 model learns how much trust should be placed in each base learner.

---



## <font color='orange'> 3. Why Use Different Algorithms? </font>

Different learning algorithms capture different aspects of the data.

For example,

| Algorithm | Strength |
| :--- | :--- |
| Logistic Regression | Linear relationships |
| Decision Tree | Simple nonlinear rules |
| Random Forest | Variance reduction |
| Gradient Boosting | Bias reduction |
| Support Vector Machine | Maximum margin classification |
| KNN | Local neighbourhood structure |

Since their errors differ,

combining them often produces better predictions.

---



## <font color='orange'> 4. General Architecture </font>

A Stacking ensemble typically follows the structure

```
Training Dataset

↓

Logistic Regression

↓

Prediction
```

```
Training Dataset

↓

Random Forest

↓

Prediction
```

```
Training Dataset

↓

Gradient Boosting

↓

Prediction
```

```
Predictions

↓

Meta-Learner

↓

Final Prediction
```

The meta-model learns how to combine the strengths of the individual learners.

---



## <font color='orange'> 5. Why Not Train the Meta-Learner Directly? </font>

One might consider training

all base learners

on the entire training dataset,

and then using their predictions to train the meta-model.

However,

this creates a serious problem.

The base learners would produce

overly optimistic predictions

because they would be predicting the same data used for training.

The meta-learner would therefore learn from

overfitted predictions.

To avoid this,

Stacking employs

**Cross-Validation**.

---



## <font color='orange'> 6. Out-of-Fold Predictions </font>

Suppose we perform

5-fold Cross-Validation.

```
Fold 1

↓

Train on Folds 2-5

↓

Predict Fold 1
```

```
Fold 2

↓

Train on Folds 1,3,4,5

↓

Predict Fold 2
```

The process continues until

every observation has been predicted by

a model that

has never seen that observation.

These predictions are called

**Out-of-Fold (OOF) Predictions**.

They form the training dataset for the meta-learner.

---



## <font color='orange'> 7. The Meta-Learner </font>

Suppose three base models produce

```
LR Prediction

RF Prediction

GB Prediction
```

The new feature vector becomes

$$
\boxed{
\mathbf z
=
(z_1,z_2,z_3),
}
$$

where

* $z_1$ is the Logistic Regression prediction,
* $z_2$ is the Random Forest prediction,
* $z_3$ is the Gradient Boosting prediction.

The meta-model then learns

$$
\boxed{
F(\mathbf z).
}
$$

Thus,

Stacking learns

**how to combine predictions**,

rather than making predictions directly from the original features.

---



## <font color='orange'> 8. Mathematical Formulation </font>

Suppose we have

$$
M
$$

base learners,

$$
f_1,f_2,\ldots,f_M.
$$

Each learner predicts

$$
f_m(\mathbf x).
$$

The meta-model

$$
g(\cdot)
$$

produces

$$
\boxed{
F(\mathbf x)
=
g
\left(
f_1(\mathbf x),
f_2(\mathbf x),
\ldots,
f_M(\mathbf x)
\right).
}
$$

Notice that

the original features

are replaced by

the predictions of the base learners.

---



## <font color='orange'> 9. Choice of Meta-Learner </font>

The meta-model is usually simple.

Common choices include

* Logistic Regression
* Ridge Regression
* Linear Regression
* Elastic Net

Simple models reduce the risk of overfitting.

Complex meta-models are possible,

but usually unnecessary.

---



## <font color='orange'> 10. Classification vs Regression </font>

For

**Classification**

base learners typically output

class probabilities.

The meta-model predicts

the final class.

---

For

**Regression**

base learners output

continuous predictions.

The meta-model predicts

the final numerical value.

---



## <font color='orange'> 11. Advantages of Stacking </font>

Stacking provides several important benefits.

* Combines strengths of different algorithms.
* Reduces both bias and variance.
* Often achieves state-of-the-art performance.
* Flexible.
* Model-agnostic.
* Can combine classifiers and regressors.

These advantages explain why Stacking is popular in data science competitions.

---



## <font color='orange'> 12. Limitations </font>

Stacking also possesses several disadvantages.

* Computationally expensive.
* Requires Cross-Validation.
* More difficult to interpret.
* Larger memory requirements.
* Higher implementation complexity.

Proper Cross-Validation is essential to avoid information leakage.

---



## <font color='orange'> 13. Hyperparameters </font>

Important design choices include

### Base Learners

```python
estimators
```

---

### Meta-Learner

```python
final_estimator
```

---

### Number of Folds

```python
cv
```

Controls the Cross-Validation used to generate Out-of-Fold predictions.

---

### Passthrough

```python
passthrough=True
```

Allows the original features to be supplied to the meta-model in addition to the predictions.

---



## <font color='orange'> 14. Scikit-Learn Implementation </font>

Classification

```python
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

base_models = [
    ("rf", RandomForestClassifier()),
    ("svm", SVC(probability=True))
]

model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5
)
```

Regression

```python
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

base_models = [
    ("rf", RandomForestRegressor()),
    ("gbr", GradientBoostingRegressor())
]

model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(),
    cv=5
)
```

---



## <font color='orange'> 15. Comparison with Other Ensemble Methods </font>

| Method | Base Learners | Training | Combination |
| :--- | :--- | :--- | :--- |
| Bagging | Same algorithm | Parallel | Average/Vote |
| Random Forest | Decision Trees | Parallel | Average/Vote |
| AdaBoost | Weak learners | Sequential | Weighted Vote |
| Gradient Boosting | Weak learners | Sequential | Additive Model |
| **Stacking** | Different algorithms | Parallel + Meta-Learning | Learned Combination |

Stacking is unique because the combination strategy itself is learned from data.

---



## <font color='orange'> 16. Applications </font>

Stacking is widely used in

* Machine Learning Competitions (Kaggle)
* Medical Diagnosis
* Financial Forecasting
* Fraud Detection
* Credit Risk Analysis
* Particle Physics
* Image Classification
* Recommendation Systems

Whenever multiple high-performing models exist,

Stacking often produces the strongest overall predictor.

---

## <font color='purple'> 17. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| Stacking | Stacked Generalization |
| Base Learners | Different machine learning algorithms |
| Meta-Learner | Learns how to combine predictions |
| Training Data | Out-of-Fold predictions |
| Cross-Validation | Prevents information leakage |
| Main Objective | Learn the optimal combination of models |
| Main Strength | Combines complementary algorithms |

> **Key Insight:** Stacking extends ensemble learning by replacing fixed aggregation rules, such as averaging or majority voting, with a learned combination strategy. Multiple base learners first generate predictions, and these predictions become the inputs to a meta-learner that discovers how to combine them most effectively. To prevent overfitting, the meta-learner is trained using out-of-fold predictions obtained through cross-validation, ensuring that every prediction is made for data unseen during the corresponding model's training. By leveraging the complementary strengths of different algorithms, Stacking often achieves predictive performance that surpasses any individual model, making it one of the most powerful and flexible ensemble learning techniques available.